<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Композиция против наследования. Внедрение зависимостей. Адаптация классических паттернов под Python</h2>
  <p>Паттерны проектирования — проверенные решения типовых задач. Но в Python многие из них выглядят иначе, чем в C++ или Java. В этой лекции вы узнаете:</p>
  <ul>
    <li>Как реализовать <strong>Singleton</strong> по‑питоньи (метакласс, декоратор, модуль)</li>
    <li>Как создать <strong>Фабрику</strong> без кучи классов (словари, callable, регистрация)</li>
    <li>Как применить <strong>Стратегию</strong> с помощью функций и объектов</li>
  </ul>
</div>

## Содержание
1. Почему наследование — это часто плохо?
2. Композиция: как строить из деталей
3. Внедрение зависимостей (DI) — дайте мне готовые объекты
4. Инверсия зависимостей через `Protocol` (абстракции вместо конкретики)
5. Контейнер зависимостей: диспетчер для ваших фабрик
6. Паттерн «Делегирование» — поручи работу другому
7. Большая практика: от зоопарка к гибкой архитектуре
8. Итоги и шпаргалка

In [ ]:
# Пример: классическая иерархия наследования (проблемная)

class Animal:
    """Базовый класс животного"""
    def __init__(self, name: str):
        self.name = name
    
    def eat(self) -> str:
        return f"{self.name} ест"

class Bird(Animal):
    """Птица — умеет летать (по задумке)"""
    def fly(self) -> str:
        return f"{self.name} летит"

class Sparrow(Bird):
    """Воробей — всё ок"""
    pass

class Penguin(Bird):
    """Пингвин — птица, но не летает! Нарушает принцип подстановки Лисков"""
    def fly(self) -> str:
        return f"{self.name} не умеет летать, только плавать"  # Костыль

# Проблемы такого подхода:
# 1. Жёсткость — нельзя переиспользовать поведение полёта для не-птиц (например, для самолёта)
# 2. Хрупкость — изменение в Bird может сломать всех потомков
# 3. Принуждение — пингвин вынужден иметь метод fly, который не должен быть

## 1. Почему глубокое наследование — зло?

Представьте, что у вас есть дедушка `Animal`, от него унаследован `Bird`, от него `FlyingBird`, потом `Sparrow`, `Eagle`… И вдруг появляется `Ostrich` (страус) или `Kiwi`. Они птицы, но не летают. А ещё есть `Bat` (летучая мышь) — не птица, но летает. 

**Главная боль:**  
Наследование фиксирует **отношение «является»** (`is-a`), но в реальном мире объекты часто обладают **поведениями**, которые не укладываются в одну «ветку» генеалогического дерева.

**Программные проблемы:**
- **Хрупкость (fragility)**: изменение в базовом классе может неожиданно сломать всех потомков.
- **Негибкость (inflexibility)**: нельзя изменить поведение объекта во время выполнения (у птицы отросли реактивные двигатели).
- **Проблема ромба (diamond problem)** при множественном наследовании.

**Решение:**  
Композиция — собираем объект из независимых «кирпичиков» поведения.

In [ ]:
# Композиция — объект содержит в себе другие объекты (поведения)

class FlyBehavior:
    """Стратегия полёта — отдельный класс"""
    def fly(self, name: str) -> str:
        return f"{name} парит в небе"

class SwimBehavior:
    """Стратегия плавания"""
    def swim(self, name: str) -> str:
        return f"{name} рассекает воду"

# Класс животного, который получает поведения извне
class AnimalComposition:
    def __init__(self, name: str, fly_behavior=None, swim_behavior=None):
        self.name = name
        self._fly = fly_behavior   # может быть None
        self._swim = swim_behavior
    
    def fly(self):
        if self._fly:
            return self._fly.fly(self.name)
        return f"{self.name} не умеет летать"
    
    def swim(self):
        if self._swim:
            return self._swim.swim(self.name)
        return f"{self.name} не умеет плавать"

# Собираем разных животных
sparrow = AnimalComposition("Воробей", FlyBehavior(), None)
penguin = AnimalComposition("Пингвин", None, SwimBehavior())
bat = AnimalComposition("Летучая мышь", FlyBehavior(), None)  # не птица, но летает

print(sparrow.fly())    # Воробей парит в небе
print(penguin.swim())   # Пингвин рассекает воду
print(bat.fly())        # Летучая мышь парит в небе

## 2. Композиция — это «has-a» (имеет)

Наследование говорит: «Воробей — это птица» (`is-a`).  
Композиция говорит: «Воробей **имеет** поведение полёта» (`has-a`).

**Плюсы композиции:**
- ✅ Гибкость — можно менять поведение в рантайме (например, заменить `FlyBehavior` на `JetFlyBehavior`).
- ✅ Переиспользование — одно поведение можно дать разным классам (человек с реактивным ранцем, летучая мышь, самолёт).
- ✅ Тестируемость — легко подставить мок-объект (заглушку) вместо настоящего полёта.
- ✅ Слабая связность — классы не зависят от конкретных реализаций (см. следующий раздел про DIP).

**Минусы:**  
- Чуть больше кода (но это окупается гибкостью).
- Нужно продумывать интерфейсы.

### Пример динамической замены поведения

In [ ]:
class RocketFly:
    def fly(self, name: str) -> str:
        return f"{name} взлетает на ракетной тяге 🚀"

# Создаём птицу с обычным полётом
bird = AnimalComposition("Орёл", FlyBehavior())
print(bird.fly())   # Орёл парит в небе

# Меняем поведение динамически (в Python это легко)
bird._fly = RocketFly()
print(bird.fly())   # Орёл взлетает на ракетной тяге 🚀

## 3. Внедрение зависимостей (Dependency Injection)

**Зависимость** — это любой объект, который нужен вашему классу для работы.  
**Внедрение зависимости** — передача этой зависимости снаружи (обычно через конструктор), а не создание её внутри.

### ❌ Плохо: жёсткая связь

In [ ]:
class EmailSender:
    def send(self, msg: str):
        print(f"Email: {msg}")

class Notification:
    def __init__(self):
        self.sender = EmailSender()   # жёсткая привязка к конкретному классу
    
    def alert(self, msg: str):
        self.sender.send(msg)

# Теперь Notification невозможно использовать с SMS или Telegram без переписывания
# И протестировать тоже трудно — всегда будет реальная отправка

In [ ]:
# ✅ Хорошо: внедрение через конструктор

class NotificationDI:
    def __init__(self, sender):   # sender — любая "штука", умеющая send
        self.sender = sender
    
    def alert(self, msg: str):
        self.sender.send(msg)

# Теперь подставляем реального отправителя
email_sender = EmailSender()
notify = NotificationDI(email_sender)
notify.alert("Привет!")

# Для теста подставляем заглушку (mock)
class MockSender:
    def __init__(self):
        self.messages = []
    def send(self, msg: str):
        self.messages.append(msg)

mock = MockSender()
test_notify = NotificationDI(mock)
test_notify.alert("Тестовое сообщение")
assert mock.messages == ["Тестовое сообщение"]
print("Тест пройден!")

# Легко переключиться на SMS
class SmsSender:
    def send(self, msg: str):
        print(f"SMS: {msg}")

notify_sms = NotificationDI(SmsSender())
notify_sms.alert("Код подтверждения")

## 4. Инверсия зависимостей (DIP) и протоколы

**Принцип DIP (Dependency Inversion Principle,** часть SOLID):  
> Модули верхнего уровня не должны зависеть от модулей нижнего уровня. И те, и другие должны зависеть от абстракций.

В Python роль абстракции играет **`typing.Protocol`** — он описывает, какие методы должен иметь объект (утиная типизация с контролем).

### Создаём протокол для отправителя

In [ ]:
from typing import Protocol

class Sender(Protocol):
    """Любой объект, который может отправить сообщение"""
    def send(self, msg: str) -> None:   # протокол требует метод send
        ...

# Теперь NotificationDI может быть аннотирован типом Sender
class BetterNotification:
    def __init__(self, sender: Sender):   # ожидаем любой класс, реализующий Protocol
        self.sender = sender
    
    def alert(self, msg: str):
        self.sender.send(msg)

# Классы, которые удовлетворяют протоколу (неявно, не нужно наследовать)
class EmailSender:
    def send(self, msg: str):
        print(f"Email: {msg}")

class TelegramSender:
    def send(self, msg: str):
        print(f"Telegram: {msg}")

# Всё работает
note = BetterNotification(EmailSender())
note.alert("Hello via protocol")

### Преимущества протоколов:
- **Статическая проверка** — IDE и `mypy` подскажут, если переданный объект не имеет метода `send`.
- **Слабая связанность** — класс `BetterNotification` не знает ничего о реальных отправителях.
- **Документация** — протокол сразу говорит, что требуется от зависимости.

## 5. Контейнер зависимостей (простой на `dict`)

Когда в программе много объектов, вручную передавать их становится утомительно.  
**Контейнер** — это «волшебный словарь», который умеет создавать объекты по запросу.

In [ ]:
class DIContainer:
    """Простейший контейнер: регистрируем фабрики, получаем объекты"""
    def __init__(self):
        self._factories = {}   # имя -> функция, создающая объект
    
    def register(self, name: str, factory):
        """Регистрируем фабрику под именем"""
        self._factories[name] = factory
    
    def resolve(self, name: str, **kwargs):
        """Создаём объект по имени, передавая kwargs в фабрику"""
        factory = self._factories.get(name)
        if not factory:
            raise KeyError(f"Сервис {name} не зарегистрирован")
        return factory(**kwargs)

# Пример: регистрируем сервисы
container = DIContainer()
container.register("email_sender", lambda: EmailSender())
container.register("sms_sender", lambda: SmsSender())
container.register("notifier", lambda: BetterNotification(container.resolve("email_sender")))

# Используем
notifier = container.resolve("notifier")
notifier.alert("Создано через контейнер!")

# Можно легко поменять зависимость, зарегистрировав новую фабрику
container.register("notifier", lambda: BetterNotification(container.resolve("sms_sender")))
notifier2 = container.resolve("notifier")
notifier2.alert("Теперь через SMS")

### Реальные фреймворки DI:
- `dependency-injector` — мощный, с иерархиями и скоупом.
- `FastAPI` имеет встроенный DI через `Depends`.

## 6. Паттерн «Делегирование» — передай ответственность

Делегирование — это когда объект **перепоручает выполнение задачи** другому объекту, но при этом сам остаётся публичным интерфейсом.

In [ ]:
class FileWriter:
    """Умеет писать в файл"""
    def write(self, data: str, filename: str):
        with open(filename, 'w') as f:
            f.write(data)

class NetworkWriter:
    """Умеет отправлять по сети"""
    def write(self, data: str, url: str):
        print(f"POST {data} to {url}")

class Logger:
    """Логгер делегирует запись конкретному writer'у"""
    def __init__(self, writer):
        self._writer = writer   # делегат
        self._destination = "log.txt"
    
    def log(self, message: str):
        # Делегируем запись
        self._writer.write(message, self._destination)

# Разные варианты использования
file_logger = Logger(FileWriter())
file_logger.log("Событие 1")

network_logger = Logger(NetworkWriter())
network_logger._destination = "http://logs.myserver.com"
network_logger.log("Событие 2")

**Почему делегирование лучше переопределения методов?**
- Не нужно создавать подкласс для каждого варианта поведения.
- Можно менять делегата в рантайме.
- Код становится проще для понимания (явная композиция).

## 7. Большая практика: от зоопарка к гибкой архитектуре

Рефакторим классический пример с животными, используя все изученные приёмы.

In [ ]:
# Исходная плохая иерархия (для сравнения)
class Bird:
    def __init__(self, name):
        self.name = name
    def eat(self): return f"{self.name} клюёт зерно"
    def fly(self): return f"{self.name} машет крыльями"

class Parrot(Bird):
    def talk(self): return f"{self.name} говорит: Привет!"

class Ostrich(Bird):
    def fly(self): return f"{self.name} бегает, не летает"   # костыль

# ------------------------- Новая гибкая архитектура -------------------------

from typing import Protocol, Optional

# 1. Определяем протоколы поведения
class Eatable(Protocol):
    def eat(self, name: str) -> str: ...

class Flyable(Protocol):
    def fly(self, name: str) -> str: ...

class Talkable(Protocol):
    def talk(self, name: str) -> str: ...

# 2. Реализуем конкретные стратегии
class SeedEat:
    def eat(self, name: str) -> str:
        return f"{name} клюёт семечки"

class InsectEat:
    def eat(self, name: str) -> str:
        return f"{name} ловит насекомых"

class NormalFly:
    def fly(self, name: str) -> str:
        return f"{name} парит в воздухе"

class NoFly:
    def fly(self, name: str) -> str:
        return f"{name} не умеет летать"

class ParrotTalk:
    def talk(self, name: str) -> str:
        return f"{name} говорит: Кто там?"

class NoTalk:
    def talk(self, name: str) -> str:
        return f"{name} молчит"

# 3. Собираем животное из стратегий
class Animal:
    def __init__(self, name: str, 
                 eat_behavior: Eatable,
                 fly_behavior: Optional[Flyable] = None,
                 talk_behavior: Optional[Talkable] = None):
        self.name = name
        self._eat = eat_behavior
        self._fly = fly_behavior
        self._talk = talk_behavior
    
    def eat(self) -> str:
        return self._eat.eat(self.name)
    
    def fly(self) -> str:
        if self._fly:
            return self._fly.fly(self.name)
        return f"{self.name} не может летать"
    
    def talk(self) -> str:
        if self._talk:
            return self._talk.talk(self.name)
        return f"{self.name} не говорит"

# 4. Создаём разных животных гибко
parrot = Animal("Попугай", SeedEat(), NormalFly(), ParrotTalk())
ostrich = Animal("Страус", SeedEat(), NoFly(), NoTalk())
bat = Animal("Летучая мышь", InsectEat(), NormalFly(), NoTalk())
fish = Animal("Рыба", InsectEat(), None, None)

# 5. Демонстрация
for animal in [parrot, ostrich, bat, fish]:
    print(f"{animal.name}: {animal.eat()}, {animal.fly()}, {animal.talk()}")

# 6. Легко написать тест, подменяя любое поведение
def test_animal_fly():
    mock_fly = lambda name: f"Mock {name} flies"
    test_animal = Animal("Тестовый", SeedEat(), mock_fly)
    assert test_animal.fly() == "Mock Тестовый flies"
    print("Тест полёта пройден")

test_animal_fly()

## 8. Итоги и шпаргалка

| Подход | Плюсы | Минусы | Когда использовать |
|--------|-------|--------|---------------------|
| **Наследование** | Простота, переиспользование кода через `super()` | Жёсткость, хрупкость, проблема ромба | Строгое `is-a`, поведение не меняется |
| **Композиция** | Гибкость, смена поведения в рантайме, тестируемость | Больше кода, нужно продумывать интерфейсы | Почти всегда, кроме тривиальных случаев |
| **DI через конструктор** | Явные зависимости, лёгкое тестирование | Код становится чуть многословнее | Всегда, когда класс использует другие объекты |
| **Протоколы** | Статическая проверка утиной типизации | Требует настройки типов | При создании абстракций для зависимостей |
| **Контейнер DI** | Упрощает создание сложных графов объектов | Может скрыть реальные зависимости | В больших приложениях (фреймворках) |
| **Делегирование** | Замена наследования без потери интерфейса | Нужно прописывать методы-обёртки | Когда нужно переиспользовать поведение без наследования |

### Запомните правила:
1. **«Предпочитайте композицию наследованию»** — не просто красивая фраза, а реальный принцип проектирования.
2. **Внедряйте зависимости, не создавайте их внутри** — код станет тестируемым и гибким.
3. **Зависьте от абстракций (протоколов), а не от конкретных классов** — это даст свободу замены реализаций.
4. **Делегирование — ваш друг** — делегируйте то, что может меняться.

